<a href="https://colab.research.google.com/github/arun-antony/ai-colab-tryout/blob/main/capstone_personalised_recommendation_retail.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Install libraries (as suggested in generic notebook)
!pip install -q sentence-transformers faiss-cpu pandas numpy

import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load datasets
products = pd.read_csv('retail_products.csv')
customers = pd.read_csv('retail_customers.csv')
events = pd.read_csv('retail_events.csv')

In [18]:
# Create a text representation for each product, this will be used for creating the embedding
products['combined_text'] = products.apply(
    lambda row: f"Product: {row['category']} by {row['brand']}", axis=1
)

print("Sample Product Description:")
print(products['combined_text'].iloc[0])

Sample Product Description:
Product: apparel by Nova


In [19]:
# Load the model from generic notebook
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all products
product_embeddings = embedding_model.encode(products['combined_text'].tolist(), show_progress_bar=True)

# Convert to float32 for FAISS compatibility
product_embeddings = np.array(product_embeddings).astype('float32')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [20]:
# Initialize FAISS index
dimension = product_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(product_embeddings)

print(f"Indexed {index.ntotal} products in vector store.")

Indexed 500 products in vector store.


In [38]:
def recommend_by_category(customer_id, k_per_category=3):
    # 1. Get user events
    user_events = events[events['customer_id'] == customer_id].copy()

    final_recommendations = {}
    seen_product_ids = set()

    # We prioritize categories: Purchase > Cart > View
    categories = ['purchase', 'cart', 'view']

    for cat in categories:
        cat_events = user_events[user_events['event_type'] == cat]

        if cat_events.empty:
            continue

        # 2. Create a "Category Vector" (Mean of product embeddings in this specific category)

        # Convert Product IDs (text) into Row Indices (integers) so we can look up their vectors in the embedding matrix
        interacted_indices = products[products['product_id'].isin(cat_events['product_id'])].index.tolist()

        # now create a list of embeddings that the user interacted with for the given category
        cat_embeddings = product_embeddings[interacted_indices]

        # Create a single 'Average Taste Vector' by calculating the mean of all product embeddings in this category;
        # Reshape to 2D and convert to float32 to meet the strict technical requirements of the FAISS vector engine.
        query_vector = np.mean(cat_embeddings, axis=0).reshape(1, -1).astype('float32')

        # 3. Search FAISS (requesting more than k to account for deduplication)
        distances, indices = index.search(query_vector, k_per_category + 5)

        # 4. Filter for uniqueness and relevance
        cat_results = []
        for idx in indices[0]:
            pid = products.iloc[idx]['product_id']
            # Only add if it's unique AND the user hasn't already interacted with it
            if pid not in seen_product_ids and pid not in user_events['product_id'].values:
                cat_results.append(products.iloc[idx])
                seen_product_ids.add(pid)

            if len(cat_results) >= k_per_category:
                break

        final_recommendations[cat] = pd.DataFrame(cat_results)

    return final_recommendations

In [1]:
# To authenticate with Hugging Face and access gated models, you need to log in.
# 1. Go to https://huggingface.co/settings/tokens to create a new token.
# 2. In Google Colab, go to the left panel (the key icon) to 'Secrets'.
# 3. Add a new secret with the name 'HF_TOKEN' and paste your Hugging Face token as the value.
# 4. Then, run the following code:

from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from Colab secrets
hf_token = userdata.get('hf-colab')

# Log in to Hugging Face
login(token=hf_token)

print("Successfully logged in to Hugging Face.")

Successfully logged in to Hugging Face.


In [3]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import torch

model_id = "google/gemma-3-4b-it"

llm = Gemma3ForConditionalGeneration.from_pretrained(
    model_id, device_map="auto"
).eval()

processor = AutoProcessor.from_pretrained(model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [30]:
def generate_explanation_with_gemma(customer_id, interaction_type, rec_df):
    # --- Data Preparation ---
    past_interactions = events[(events['customer_id'] == customer_id) &
                               (events['event_type'] == interaction_type)]
    past_products = products[products['product_id'].isin(past_interactions['product_id'])]

    history_str = ", ".join([f"{r.brand} {r.category}" for _, r in past_products.iterrows()])
    rec_str = ", ".join([f"{r.brand} {r.category}" for _, r in rec_df.iterrows()])

    print(f"{interaction_type} History: {history_str}")
    print(f"Recommendations: {rec_str}")
    # --- Gemma 3 Formatting ---
    messages = [
        {"role": "system", "content": f"You are a helpful retail assistant. Explain recommendations clearly based on {interaction_type} history."},
        {"role": "user", "content": f"The customer has a history of {interaction_type}s: {history_str}. Explain why they might like these new items: {rec_str}."}
    ]

    # Process the prompt
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=prompt, return_tensors="pt").to(llm.device)

    # Generate
    output_tokens = llm.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.1
    )

    # Decode only the new generated text
    new_tokens = output_tokens[0][len(inputs.input_ids[0]):]
    return processor.decode(new_tokens, skip_special_tokens=True)

In [37]:
test_rec = recommend_by_category("C000467")
print(test_rec["view"])

# for cat, df in test_rec.items():
#     print(f"\n--- Recommendations based on your {cat.upper()}S ---")
#     if not df.empty:
#         display(df[['product_id', 'category', 'brand', 'price']])
#     else:
#         print("No recommendations for this category.")

   product_id category brand  price            combined_text
8      P00008   beauty  Nova  12.23  Product: beauty by Nova
11     P00011   beauty  Nova  11.04  Product: beauty by Nova
29     P00029   beauty  Nova  12.74  Product: beauty by Nova


In [32]:
result = generate_explanation_with_gemma("C000467", "cart", test_rec["cart"])
print(result)

cart History: Acme electronics
Recommendations: Acme electronics, Acme electronics, Acme electronics
Okay, let’s take a look at your shopping history! I see you’ve purchased items from Acme Electronics quite a few times – three times in fact! That tells me you’re clearly interested in electronics, and specifically, you seem to be drawn to products from Acme. 

Based on that, I’m going to recommend a few things I think you’ll really enjoy. Since you’ve bought Acme products before, we can assume you appreciate their quality and features. 

Here are a few suggestions:

* **[New Product 1 - e.g., Acme Wireless Charging Pad]:** "Since you've previously purchased Acme products, you're probably looking for convenient charging solutions. This wireless charging pad
